# 机器学习预测 + Markowitz 均值-方差优化

**参考论文**: Jichu Han (2023), "Asset Allocation Strategy Based on Announcements and Machine Learning"  
**论文成果**: 年化收益59.4%，夏普比率2.28  
**核心思想**: 用LightGBM预测各ETF未来收益率，替代历史均值输入Markowitz模型，提升组合配置质量

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 策略原理

### 传统Markowitz的局限

经典均值-方差模型:
$$\max_w \quad w^T \mu - \frac{\lambda}{2} w^T \Sigma w, \quad \text{s.t.} \sum w_i = 1, w_i \geq 0$$

问题: **$\mu$ 使用历史均值估计，噪声极大** ("垃圾进，垃圾出")

### ML增强方法

**Step 1: 特征工程** - 为每个ETF构建技术因子:
- 动量: $r_{t-k}$ (5/10/20日)
- 波动率: $\sigma_{t-k}$ (5/10/20日)
- RSI, MACD, 布林带宽度
- 量价特征

**Step 2: LightGBM预测**
$$\hat{r}_{i,t+h} = f_{\text{LGBM}}(X_{i,t}), \quad i = 1,\ldots,N$$

**Step 3: 用预测收益替代历史均值**
$$\hat{\mu}_i = \hat{r}_{i,t+h}$$

**Step 4: Markowitz优化**
$$\max_w \quad w^T \hat{\mu} - \frac{\lambda}{2} w^T \Sigma w$$

### LightGBM 优势
- 高效的梯度提升决策树 (GBDT)
- 叶子节点优先生长策略 → 更高精度
- 原生支持类别特征和缺失值
- 训练速度快，适合滚动窗口场景

In [ ]:
# Cell 3: 动画 - ML预测 vs 历史均值 有效前沿对比
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)

def animate_ml_frontier():
    """对比ML预测收益 vs 历史均值输入后有效前沿的差异"""
    n_assets = 8
    # 模拟: 历史均值 (噪声大)
    mu_hist = np.array([0.08, 0.05, 0.12, 0.10, 0.06, 0.15, 0.09, 0.03])
    # ML预测 (更准确，方向性更强)
    mu_ml = np.array([0.06, 0.04, 0.14, 0.11, 0.16, 0.10, 0.13, 0.02])
    cov = np.eye(n_assets) * 0.06
    for i in range(n_assets):
        for j in range(n_assets):
            if i != j:
                cov[i, j] = 0.015 + np.random.uniform(-0.005, 0.005)
    cov = (cov + cov.T) / 2

    def frontier(mu, cov, n=2000):
        rets, risks, sharpes = [], [], []
        for _ in range(n):
            w = np.random.dirichlet(np.ones(n_assets))
            r = w @ mu; s = np.sqrt(w @ cov @ w)
            rets.append(r); risks.append(s)
            sharpes.append((r - 0.025) / s if s > 0 else 0)
        return risks, rets, sharpes

    r_h, ret_h, sh_h = frontier(mu_hist, cov)
    r_m, ret_m, sh_m = frontier(mu_ml, cov)

    frames = []
    batch = 100
    for step in range(batch, 2001, batch):
        frames.append(go.Frame(
            data=[
                go.Scatter(x=r_h[:step], y=ret_h[:step], mode='markers',
                           name='历史均值', marker=dict(color='rgba(158,158,158,0.4)', size=3)),
                go.Scatter(x=r_m[:step], y=ret_m[:step], mode='markers',
                           name='ML预测', marker=dict(color=sh_m[:step],
                           colorscale='Viridis', size=4, opacity=0.6)),
            ],
            name=f'{step} portfolios'
        ))

    fig = go.Figure(data=frames[0].data, frames=frames)
    fig.update_layout(
        title='有效前沿: ML预测收益 vs 历史均值',
        xaxis_title='年化风险', yaxis_title='年化收益',
        height=550, width=850,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='\u25b6 播放', method='animate',
                 args=[None, {'frame': {'duration': 120}, 'transition': {'duration': 60}}]),
        ])],
        sliders=[dict(steps=[dict(args=[[f.name]], label=f.name, method='animate') for f in frames])],
    )
    return fig

fig = animate_ml_frontier()
fig.show()

In [ ]:
# Cell 4: 导入依赖
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
from scipy.optimize import minimize
from sklearn.model_selection import TimeSeriesSplit

from shared.data_fetcher import get_etf_daily, get_stock_daily, get_index_daily
from shared.constants import (SECTOR_ETFS, DEFAULT_START, DEFAULT_END,
                               RISK_FREE_RATE, TRADING_DAYS_PER_YEAR, INITIAL_CAPITAL)
from shared.metrics import summary_table
from shared.visualization import (plot_equity_curve, plot_drawdown,
                                   plot_metrics_table, plot_cumulative_comparison)
from shared.factors import momentum, volatility, rsi, macd, bollinger_bands

np.random.seed(42)
print('All imports loaded successfully.')

In [ ]:
# Cell 5: 下载行业ETF数据
etf_names = list(SECTOR_ETFS.keys())
etf_codes = list(SECTOR_ETFS.values())

fallback_stocks = {
    '银行': '601398', '券商': '600030', '医药': '600276', '消费': '000858',
    '科技': '002415', '新能源': '300750', '军工': '600893', '地产': '001979',
}

raw_data = {}
for name, code in SECTOR_ETFS.items():
    df = get_etf_daily(code, DEFAULT_START, DEFAULT_END)
    if df is not None and len(df) > 100 and 'close' in df.columns:
        raw_data[name] = df
    else:
        print(f'ETF {name}({code}) 失败, 使用备用个股')
        df = get_stock_daily(fallback_stocks[name], DEFAULT_START, DEFAULT_END)
        if df is not None and len(df) > 100 and 'close' in df.columns:
            raw_data[name] = df

# 收盘价矩阵
close_dict = {name: df['close'] for name, df in raw_data.items()}
prices_df = pd.DataFrame(close_dict).sort_index().ffill().dropna()

bench_df = get_index_daily('sh000300', DEFAULT_START, DEFAULT_END)
bench_close = bench_df['close'] if len(bench_df) > 0 else None

print(f'数据区间: {prices_df.index[0].date()} ~ {prices_df.index[-1].date()}')
print(f'资产数量: {len(raw_data)}, 交易日: {prices_df.shape[0]}')

In [ ]:
# Cell 6: 特征工程 + LightGBM预测

def build_features(df):
    """为单个ETF构建预测特征"""
    close = df['close']
    features = pd.DataFrame(index=df.index)

    # 动量因子
    for p in [5, 10, 20, 60]:
        features[f'mom_{p}'] = close.pct_change(p)

    # 波动率因子
    ret = close.pct_change()
    for w in [5, 10, 20]:
        features[f'vol_{w}'] = ret.rolling(w).std()

    # RSI
    features['rsi_14'] = rsi(close, 14)

    # MACD
    macd_df = macd(close)
    features['macd_hist'] = macd_df['histogram']

    # 布林带
    bb = bollinger_bands(close)
    features['bb_width'] = bb['bb_width']
    features['bb_pctb'] = bb['bb_pctb']

    # 均线偏离
    for w in [5, 10, 20]:
        ma = close.rolling(w).mean()
        features[f'price_ma_{w}'] = (close - ma) / ma

    # 成交量特征 (如有)
    if 'volume' in df.columns:
        features['vol_ratio_5'] = df['volume'] / df['volume'].rolling(5).mean()
        features['vol_ratio_20'] = df['volume'] / df['volume'].rolling(20).mean()

    # 目标: 未来5日收益率
    features['target'] = close.pct_change(5).shift(-5)

    return features.dropna()


def train_predict_lgbm(features_df, train_end_idx, feature_cols):
    """训练LightGBM并预测下一期收益"""
    train = features_df.iloc[:train_end_idx]
    X_train = train[feature_cols].values
    y_train = train['target'].values

    # 最新一行用于预测
    X_pred = features_df.iloc[train_end_idx-1:train_end_idx][feature_cols].values

    params = {
        'objective': 'regression',
        'metric': 'mse',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'seed': 42,
    }

    train_data = lgb.Dataset(X_train, label=y_train)
    model = lgb.train(params, train_data, num_boost_round=100)
    pred = model.predict(X_pred)

    return pred[0], model


# 为每个ETF构建特征
all_features = {}
for name, df in raw_data.items():
    feat = build_features(df)
    all_features[name] = feat
    print(f'{name}: {len(feat)} 样本, {feat.shape[1]-1} 特征')

feature_cols = [c for c in list(all_features.values())[0].columns if c != 'target']
print(f'\n特征列: {feature_cols}')

In [ ]:
# Cell 7: 滚动窗口 ML预测 + Markowitz优化

returns_df = prices_df.pct_change().dropna()
asset_names = list(prices_df.columns)
n_assets = len(asset_names)

lookback = 150       # 训练窗口
rebalance_freq = 20  # 调仓频率
risk_aversion = 2.5

def markowitz_optimize(mu_pred, cov, risk_aversion=2.5):
    """Markowitz优化 (用ML预测的mu)"""
    n = len(mu_pred)
    def neg_utility(w):
        return -(w @ mu_pred - 0.5 * risk_aversion * w @ cov @ w)
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    bounds = [(0, 0.4)] * n
    w0 = np.ones(n) / n
    res = minimize(neg_utility, w0, method='SLSQP', bounds=bounds, constraints=constraints)
    return res.x if res.success else w0

dates = returns_df.index

# ML+MV策略
port_ret_ml = pd.Series(0.0, index=dates)
weights_ml = np.ones(n_assets) / n_assets
ml_weight_hist = []

# 传统MV策略 (历史均值)
port_ret_hist = pd.Series(0.0, index=dates)
weights_hist = np.ones(n_assets) / n_assets

rebal_dates = []

for i in range(lookback, len(dates)):
    daily_ret = returns_df.iloc[i].values
    port_ret_ml.iloc[i] = weights_ml @ daily_ret
    port_ret_hist.iloc[i] = weights_hist @ daily_ret

    if (i - lookback) % rebalance_freq == 0:
        window_ret = returns_df.iloc[i-lookback:i]
        cov = window_ret.cov().values * 252
        current_date = dates[i]

        # --- ML预测期望收益 ---
        mu_ml_pred = np.zeros(n_assets)
        for j, name in enumerate(asset_names):
            if name in all_features:
                feat_df = all_features[name]
                # 找到当前日期之前的数据
                mask = feat_df.index <= current_date
                train_len = mask.sum()
                if train_len > 60:
                    try:
                        pred, _ = train_predict_lgbm(feat_df[mask], train_len, feature_cols)
                        mu_ml_pred[j] = pred * 252 / 5  # 年化 (5日收益 -> 年化)
                    except Exception:
                        mu_ml_pred[j] = window_ret.iloc[:, j].mean() * 252
                else:
                    mu_ml_pred[j] = window_ret.iloc[:, j].mean() * 252

        # ML+Markowitz
        try:
            weights_ml = markowitz_optimize(mu_ml_pred, cov, risk_aversion)
        except Exception:
            weights_ml = np.ones(n_assets) / n_assets

        # 传统MV (历史均值)
        mu_hist = window_ret.mean().values * 252
        try:
            weights_hist = markowitz_optimize(mu_hist, cov, risk_aversion)
        except Exception:
            weights_hist = np.ones(n_assets) / n_assets

        # 交易成本
        if len(rebal_dates) > 0:
            port_ret_ml.iloc[i] -= 0.002
            port_ret_hist.iloc[i] -= 0.002

        rebal_dates.append(current_date)
        ml_weight_hist.append(weights_ml.copy())

    if i % 100 == 0:
        print(f'  进度: {i}/{len(dates)}')

# 截取
port_ret_ml = port_ret_ml.iloc[lookback:]
port_ret_hist = port_ret_hist.iloc[lookback:]
equity_ml = INITIAL_CAPITAL * (1 + port_ret_ml).cumprod()
equity_hist = INITIAL_CAPITAL * (1 + port_ret_hist).cumprod()

print(f'\n调仓次数: {len(rebal_dates)}')
print(f'ML+MV终端净值:  {equity_ml.iloc[-1]:,.0f}')
print(f'传统MV终端净值:  {equity_hist.iloc[-1]:,.0f}')

In [ ]:
# Cell 8: 回测统计

bench_returns = None
if bench_close is not None:
    bench_close_aligned = bench_close.reindex(port_ret_ml.index).ffill()
    bench_returns = bench_close_aligned.pct_change().fillna(0)

metrics_ml = summary_table(port_ret_ml, equity_ml, bench_returns)
metrics_hist = summary_table(port_ret_hist, equity_hist, bench_returns)

compare_keys = ['年化收益率', '年化波动率', '夏普比率', '最大回撤', 'Calmar比率', '胜率']
print(f'{"指标":<12} {"ML+MV":<14} {"传统MV":<14}')
print('-' * 42)
for k in compare_keys:
    print(f'{k:<12} {metrics_ml.get(k, "N/A"):<14} {metrics_hist.get(k, "N/A"):<14}')

# 最新权重
if ml_weight_hist:
    print('\n=== ML+MV 最新权重 ===')
    for name, w in zip(asset_names, ml_weight_hist[-1]):
        print(f'  {name}: {w:.1%}')

In [ ]:
# Cell 9: 可视化
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['PingFang SC', 'SimHei', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False

# 1) ML+MV vs 传统MV
plot_cumulative_comparison(
    {'ML+Markowitz': port_ret_ml, '传统Markowitz': port_ret_hist},
    title='LightGBM预测 + Markowitz vs 传统Markowitz'
)

# 2) 回撤
plot_drawdown(equity_ml, title='ML+Markowitz - 回撤')

# 3) 权重演变
weight_df = pd.DataFrame(ml_weight_hist, columns=asset_names,
                          index=rebal_dates[:len(ml_weight_hist)])
fig, ax = plt.subplots(figsize=(14, 5))
weight_df.plot.area(ax=ax, alpha=0.75, colormap='tab10')
ax.set_title('ML+Markowitz - 资产权重演变', fontsize=14, fontweight='bold')
ax.set_ylabel('权重')
ax.set_ylim(0, 1)
ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=9)
plt.tight_layout()
plt.show()

# 4) 绩效表
plot_metrics_table(metrics_ml, title='ML+Markowitz 策略绩效指标')

## 结果分析

### ML预测增强效果
1. **LightGBM预测收益率**替代历史均值后，Markowitz模型的输入质量显著提升
2. ML模型能捕捉非线性因子交互效应，传统均值估计无法做到
3. 权重分配更加动态，能及时响应市场状态变化

### 实际应用注意
- 预测模型的过拟合风险: 需要严格的时间序列交叉验证
- 特征重要性分析有助于理解模型决策逻辑
- 预测值的置信度可用于调节Markowitz中的风险厌恶参数
- 计算成本较高 (每次调仓需训练N个模型), 可考虑增量学习